# Biapy Colab Training

In [1]:
!pwd

In [2]:
!pwd
!ls
from google.colab import drive
drive.mount('/content/drive')

In [3]:
!git clone https://github.com/NovaBro/neuroinfo_fruitfly.git
%ls
%cd neuroinfo_fruitfly
%pwd
%ls

In [4]:
!chmod +x fisbe.sh
!./fisbe.sh

In [5]:
!cat download.txt

In [6]:
import time
from pathlib import Path
from IPython.display import clear_output

MAX_SIZE = 0
while MAX_SIZE < 20000:
    clear_output(wait=True)
    fisbe = Path("fisbe")
    if not fisbe.exists():
        print("fisbe/ does not exist yet")
    else:
        for path in sorted(fisbe.iterdir()):
            size_mb = path.stat().st_size / (1024**2)
            MAX_SIZE = size_mb
            print(f"{path.name:40s} {size_mb:8.1f} MB")
    time.sleep(1)


In [9]:
!pip install zarr biapy

In [10]:
import zarr
import shutil
from tifffile import imwrite
from tqdm import tqdm
import numpy as np
import os

## Create tiff Version for BiaPy

In [11]:
DATASET = Path('./fisbe/completely').resolve()
train_path = DATASET / 'train'
test_path = DATASET / 'test'
val_path = DATASET / 'val'
file_name = "R38F04-20181005_63_G3.zarr"

# Create x_y_dir for BiaPy
def create_x_y_dir(path:Path):
    (path / 'raw').mkdir(parents=True, exist_ok=True)
    (path / 'label').mkdir(parents=True, exist_ok=True)

def create_dirs(root_dir:Path, dirs: list[Path]):
    for dir in dirs:
        (root_dir / dir).mkdir(parents=True, exist_ok=True)
        create_x_y_dir(root_dir / dir)

TIFF_PATH = Path('./fisbe/biapy').resolve()
shutil.rmtree(TIFF_PATH, ignore_errors=True)
TIFF_PATH.mkdir(parents=True, exist_ok=True)
create_dirs(TIFF_PATH, ['train', 'test', 'val'])
print(DATASET)
print(TIFF_PATH)

In [31]:
!unzip fisbe/fisbe_v1.0_completely.zip -d fisbe
!unzip fisbe/fisbe_v1.0_mips.zip -d fisbe
!unzip fisbe/fisbe_v1.0_partly.zip -d fisbe

In [32]:
!rm -rf completely
!rm -rf mips
!rm -rf partly

In [33]:
!ls fisbe
!ls

In [34]:
!ls /content/neuroinfo_fruitfly/fisbe/completely/train

In [35]:
raw = zarr.open(DATASET /  'train' / 'R38F04-20181005_63_G3.zarr', mode='r', path="volumes/raw")
seg = zarr.open(DATASET /  'train' / 'R38F04-20181005_63_G3.zarr', mode='r', path="volumes/gt_instances")
raw_np = np.array(raw)
seg_np = np.array(seg)
raw_np.shape, seg_np.shape, np.unique(seg_np)

In [ ]:
train_files = os.listdir(train_path)
test_files = os.listdir(test_path)
val_files = os.listdir(val_path)

def create_tiff_version(data_path, tiff_path=TIFF_PATH):
    for file in tqdm(os.listdir(data_path)):
        raw = zarr.open(data_path / file, mode='r', path="volumes/raw")
        seg = zarr.open(data_path / file, mode='r', path="volumes/gt_instances")
        raw_np = np.array(raw)
        seg_np = np.array(seg)  
        raw_np.shape, seg_np.shape, np.unique(seg_np)
        imwrite(tiff_path / 'raw' / f"{file}.tiff", raw_np)
        imwrite(tiff_path / 'label' / f"{file}_seg.tiff", seg_np)

# Create tiff version for BiaPy
for split in ['train', 'test', 'val']:
    tqdm.write(f"Running Split {split}")
    create_tiff_version(DATASET / split, TIFF_PATH / split)


In [ ]:
!python fisbe/biapy/prepare_tiff_data.py --splits test --clean
# add train val to --splits when you have those zarr splits ready